# Matching v2 - pairwise compatibility + casual sectioned note

`compatibility(a, b)` returns a grounded dict and `print_report()` renders the data view.
On top of that, `matchmaker_note(a, b)` + `render_note()` produce a casual, you-focused note to
candidate A, split into small sub-paragraphs under headers (Common ground / Friction / Your type? /
Heads-up / Verdict) via one grounded Ollama call.

Rules:
- interest sentiment: positive = love_it + cool_with, negative = dislike + hate (meh = neutral).
- preference fit is directional: for each trait a person most values, look at the *other's*
  self-rating (MET >=7 / PARTIAL 5-6 / GAP <5). For `shared_interests`, the shared-hobby overlap
  is the evidence.
- same-race / same-religion only count as factors when that person's importance is very high (>=8).
- religion has no field in the data, only an importance score -> flagged but unverifiable.
- the note rephrases ONLY the computed facts (grounded); guards reject invented brands, ids/names,
  and first-person-plural ("we/us/our").

In [1]:
# --- Config + load + helpers ---
import json

PROFILES_PATH = "profiles_ministral_3_8b.json"   # any fully-populated profiles_*.json (523 users)
IMPORTANCE_THRESHOLD = 8                          # race/religion matter only at >= this

PROFILES = json.load(open(PROFILES_PATH, encoding="utf-8"))
print("loaded", len(PROFILES), "profiles from", PROFILES_PATH)

HOBBIES = [
    "sports", "tvsports", "exercise", "dining", "museums", "art", "hiking", "gaming",
    "clubbing", "reading", "tv", "theater", "movies", "concerts", "music", "shopping", "yoga",
]
HOBBY_LABEL = {
    "sports": "playing sports", "tvsports": "watching sports", "exercise": "exercising",
    "dining": "dining out", "museums": "museums & galleries", "art": "art",
    "hiking": "hiking & camping", "gaming": "gaming", "clubbing": "dancing & clubbing",
    "reading": "reading", "tv": "watching TV", "theater": "theater", "movies": "movies",
    "concerts": "concerts", "music": "music", "shopping": "shopping", "yoga": "yoga & meditation",
}

def get(iid):
    return PROFILES[str(int(iid))]

def label(h):
    return HOBBY_LABEL.get(h, h)

def positives(p):
    bs = p["interests"]["by_sentiment"]
    return set(bs["love_it"]) | set(bs["cool_with"])

def negatives(p):
    bs = p["interests"]["by_sentiment"]
    return set(bs["dislike"]) | set(bs["hate"])

def _level(score):
    if score is None:
        return "unknown"
    if score >= 7:
        return "MET"
    if score >= 5:
        return "PARTIAL"
    return "GAP"

loaded 523 profiles from profiles_ministral_3_8b.json


In [2]:
# --- Compatibility logic ---
def _pref_fit(wanter, offerer, shared_interest_count):
    """For each trait `wanter` most values, how does `offerer` measure up?"""
    out = []
    for trait in wanter["partner_preferences"]["most_valued"]:
        if trait == "shared_interests":
            lvl = "MET" if shared_interest_count >= 3 else ("PARTIAL" if shared_interest_count >= 1 else "GAP")
            out.append({"trait": "shared_interests",
                        "evidence": f"{shared_interest_count} shared interests", "level": lvl})
        else:
            sc = offerer["self_perception"].get(trait)
            out.append({"trait": trait, "offers": sc, "level": _level(sc)})
    return out

def _constraints(a, b):
    """Race/religion factors, surfaced only when a person's importance is very high (>=8)."""
    notes = []
    for who, p, other in (("A", a, b), ("B", b, a)):
        imp_race = p["partner_importance"]["same_race"]
        if imp_race is not None and imp_race >= IMPORTANCE_THRESHOLD:
            same = p["demographics"]["race"] == other["demographics"]["race"]
            notes.append({"person": who, "factor": "same_race", "importance": imp_race,
                          "status": "match" if same else "mismatch",
                          "detail": f'{p["demographics"]["race"]} vs {other["demographics"]["race"]}'})
        imp_rel = p["partner_importance"]["same_religion"]
        if imp_rel is not None and imp_rel >= IMPORTANCE_THRESHOLD:
            notes.append({"person": who, "factor": "same_religion", "importance": imp_rel,
                          "status": "unverifiable", "detail": "no religion field in dataset"})
    return notes

def compatibility(iid_a, iid_b):
    a, b = get(iid_a), get(iid_b)
    pos_a, pos_b = positives(a), positives(b)
    neg_a, neg_b = negatives(a), negatives(b)
    ordered = lambda s: [h for h in HOBBIES if h in s]

    shared_interests = ordered(pos_a & pos_b)
    shared_dislikes = ordered(neg_a & neg_b)
    clashes = []
    for h in HOBBIES:
        if h in pos_a and h in neg_b:
            clashes.append({"hobby": h, "a": "into", "b": "dislikes"})
        elif h in neg_a and h in pos_b:
            clashes.append({"hobby": h, "a": "dislikes", "b": "into"})

    sic = len(shared_interests)
    da, db = a["demographics"], b["demographics"]
    age_gap = None
    if da["age"] is not None and db["age"] is not None:
        age_gap = abs(da["age"] - db["age"])

    return {
        "pair": {"a": int(iid_a), "b": int(iid_b),
                 "a_field": da["field_of_study"], "b_field": db["field_of_study"],
                 "genders": (da["gender"], db["gender"])},
        "shared_interests": shared_interests,
        "shared_dislikes": shared_dislikes,
        "interest_clashes": clashes,
        "a_wants_from_b": _pref_fit(a, b, sic),
        "b_wants_from_a": _pref_fit(b, a, sic),
        "shared_top_values": [t for t in a["partner_preferences"]["most_valued"]
                              if t in b["partner_preferences"]["most_valued"]],
        "constraints": _constraints(a, b),
        "age_gap": age_gap,
        "goals": (a["dating_context"]["goal"], b["dating_context"]["goal"]),
    }

In [3]:
# --- Readable report ---
def print_report(r):
    pa, pb = r["pair"]["a"], r["pair"]["b"]
    ga, gb = r["pair"]["genders"]
    print(f"COMPATIBILITY  #{pa} ({ga}, {r['pair']['a_field']})  x  #{pb} ({gb}, {r['pair']['b_field']})")
    print("-" * 70)
    print("Shared interests :", ", ".join(label(h) for h in r["shared_interests"]) or "(none)")
    print("Shared dislikes  :", ", ".join(label(h) for h in r["shared_dislikes"]) or "(none)")
    if r["interest_clashes"]:
        print("Interest clashes :")
        for c in r["interest_clashes"]:
            print(f"   - {label(c['hobby'])}: #{pa} {c['a']} / #{pb} {c['b']}")
    else:
        print("Interest clashes : (none)")

    def fmt_fit(items, offerer_id):
        for it in items:
            if it["trait"] == "shared_interests":
                print(f"   - shared interests -> {it['evidence']}  [{it['level']}]")
            else:
                off = it["offers"]
                off = f"{off:.0f}/10" if off is not None else "n/a"
                print(f"   - {it['trait']} -> #{offerer_id} rates self {off}  [{it['level']}]")

    print(f"What #{pa} values, how #{pb} measures up:")
    fmt_fit(r["a_wants_from_b"], pb)
    print(f"What #{pb} values, how #{pa} measures up:")
    fmt_fit(r["b_wants_from_a"], pa)
    print("Shared top values:", ", ".join(t.replace("_", " ") for t in r["shared_top_values"]) or "(none)")

    if r["age_gap"] is not None:
        print(f"Age gap          : {r['age_gap']:.0f} years")
    for n in r["constraints"]:
        who = pa if n["person"] == "A" else pb
        print(f"Constraint       : #{who} values {n['factor']} ({n['importance']:.0f}/10) "
              f"-> {n['status'].upper()} ({n['detail']})")
    print(f"Goals            : #{pa} {r['goals'][0]!r} / #{pb} {r['goals'][1]!r}")

In [4]:
# --- Demo on a few real opposite-gender pairs ---
females = [int(k) for k, v in PROFILES.items() if v["demographics"]["gender"] == "Female"]
males = [int(k) for k, v in PROFILES.items() if v["demographics"]["gender"] == "Male"]
demo_pairs = [(females[0], males[0]), (females[1], males[5]), (females[3], males[10])]

for a, b in demo_pairs:
    print_report(compatibility(a, b))
    print()

COMPATIBILITY  #1 (Female, Law)  x  #11 (Male, Business / Econ / Finance)
----------------------------------------------------------------------
Shared interests : playing sports, dining out, reading, movies, concerts, music
Shared dislikes  : theater, yoga & meditation
Interest clashes :
   - watching sports: #1 dislikes / #11 into
   - exercising: #1 into / #11 dislikes
   - museums & galleries: #1 dislikes / #11 into
   - watching TV: #1 into / #11 dislikes
What #1 values, how #11 measures up:
   - sincere -> #11 rates self 9/10  [MET]
   - intelligent -> #11 rates self 8/10  [MET]
   - attractive -> #11 rates self 8/10  [MET]
What #11 values, how #1 measures up:
   - attractive -> #1 rates self 6/10  [PARTIAL]
   - sincere -> #1 rates self 8/10  [MET]
   - intelligent -> #1 rates self 8/10  [MET]
Shared top values: sincere, intelligent, attractive
Age gap          : 6 years
Goals            : #1 'to meet new people' / #11 'a fun night out'

COMPATIBILITY  #2 (Female, Law)  x  #16 (

In [5]:
# --- Verification: spot-checks + the >=8 threshold rule ---
# 1) every shared interest is positive for BOTH; every shared dislike negative for both; clashes mixed.
bad = []
for a, b in demo_pairs:
    r = compatibility(a, b)
    pa, pb = get(a), get(b)
    for h in r["shared_interests"]:
        if not (h in positives(pa) and h in positives(pb)):
            bad.append(("shared_interest", a, b, h))
    for h in r["shared_dislikes"]:
        if not (h in negatives(pa) and h in negatives(pb)):
            bad.append(("shared_dislike", a, b, h))
    for c in r["interest_clashes"]:
        h = c["hobby"]
        mixed = (h in positives(pa)) != (h in positives(pb))
        if not mixed:
            bad.append(("clash", a, b, h))
print("correctness violations (want none):", bad or "none")

# 2) threshold rule: find a pair where one cares about race (>=8) and the other doesn't, show asym.
hi = next((int(k) for k, v in PROFILES.items()
           if (v["partner_importance"]["same_race"] or 0) >= IMPORTANCE_THRESHOLD
           and v["demographics"]["gender"] == "Female"), None)
lo = next((int(k) for k, v in PROFILES.items()
           if (v["partner_importance"]["same_race"] or 0) < IMPORTANCE_THRESHOLD
           and v["demographics"]["gender"] == "Male"), None)
if hi is not None and lo is not None:
    print(f"\nThreshold demo: #{hi} (cares about race) x #{lo} (does not)")
    print_report(compatibility(hi, lo))

correctness violations (want none): none

Threshold demo: #3 (cares about race) x #11 (does not)
COMPATIBILITY  #3 (Female, Math)  x  #11 (Male, Business / Econ / Finance)
----------------------------------------------------------------------
Shared interests : watching sports, dining out, reading, movies, concerts
Shared dislikes  : (none)
Interest clashes :
   - playing sports: #3 dislikes / #11 into
   - exercising: #3 into / #11 dislikes
   - watching TV: #3 into / #11 dislikes
   - theater: #3 into / #11 dislikes
   - yoga & meditation: #3 into / #11 dislikes
What #3 values, how #11 measures up:
   - attractive -> #11 rates self 8/10  [MET]
   - intelligent -> #11 rates self 8/10  [MET]
   - sincere -> #11 rates self 9/10  [MET]
What #11 values, how #3 measures up:
   - attractive -> #3 rates self 8/10  [MET]
   - sincere -> #3 rates self 9/10  [MET]
   - intelligent -> #3 rates self 9/10  [MET]
Shared top values: attractive, intelligent, sincere
Age gap          : 2 years
Constra

In [6]:
# --- Casual note layer: A-POV fact brief + grounded Ollama call (sectioned) ---
import urllib.request, re

OLLAMA_URL = "http://localhost:11434/api/chat"
NOTE_MODEL = "llama3.1:8b"
NOTE_OPTS = {"temperature": 0.85, "top_p": 0.9, "repeat_penalty": 1.2}

# the note is split into small casual sub-paragraphs, one per key:
SECTIONS = ["common_ground", "friction", "fit", "heads_up", "verdict"]
SECTION_LABELS = {
    "common_ground": "Common ground", "friction": "Friction",
    "fit": "Your type?", "heads_up": "Heads-up", "verdict": "Verdict",
}

# brands/media the model tends to invent at higher temp -> trigger a regenerate (not shown in prompt)
BANNED_NAMES = [
    "netflix", "spotify", "youtube", "instagram", "tiktok", "tinder", "hinge", "bumble",
    "facebook", "twitter", "snapchat", "disney", "marvel", "hbo", "amazon", "starbucks",
    "the office", "game of thrones", "harry potter", "star wars", "taylor swift",
]
# the friend is an outside observer, never on the date -> reject first-person-plural
_BAD_PRON = re.compile(r"\b(we|us|our|ours|we're|we've|we'd|let's)\b", re.I)
PRONOUN = {"Male": ("he", "him"), "Female": ("she", "her")}

def sanitize(s):
    s = s.replace("�", "")
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    return s.strip('"')

def _dealbreakers(r):
    """A-POV heads-up sentences (goal/age/background) - shared by the brief and the fallback."""
    subj = PRONOUN.get(r["pair"]["genders"][1], ("they", "them"))[0]
    out = []
    for n in r["constraints"]:
        if n["person"] != "A":
            continue
        if n["factor"] == "same_race":
            out.append(f"You strongly want the same background, and that's a {n['status']} here.")
        elif n["factor"] == "same_religion":
            out.append("You strongly want a shared religion, but there's no data to confirm it.")
    if r["age_gap"] is not None and r["age_gap"] >= 8:
        out.append(f"There's about a {r['age_gap']:.0f}-year age gap.")
    ga, gg = r["goals"]
    if ga != gg and "Unknown" not in (ga, gg):
        out.append(f"You're after '{ga}' while {subj} wants '{gg}'.")
    return out

def match_brief(r):
    """Compact A-point-of-view fact brief built ONLY from the compatibility dict."""
    gb = r["pair"]["genders"][1]
    subj, obj = PRONOUN.get(gb, ("they", "them"))
    L = lambda hs: ", ".join(label(h) for h in hs) or "none"
    you_love_x_hates = [c["hobby"] for c in r["interest_clashes"] if c["a"] == "into"]
    x_loves_you_hate = [c["hobby"] for c in r["interest_clashes"] if c["a"] == "dislikes"]
    fit = r["a_wants_from_b"]
    strong = [it["trait"].replace("_", " ") for it in fit if it["level"] == "MET"]
    gap = [it["trait"].replace("_", " ") for it in fit if it["level"] == "GAP"]

    lines = [
        f"The other person is {gb.lower()}; refer to {obj} as '{subj}'.",
        f"Interests you and {subj} both enjoy: {L(r['shared_interests'])}.",
        f"Things YOU like but {subj} does NOT: {L(you_love_x_hates)}.",
        f"Things {subj.upper()} likes but YOU do NOT: {L(x_loves_you_hate)}.",
        f"Things you and {subj} both can't stand: {L(r['shared_dislikes'])}.",
        f"Traits you want that {subj} is strong on: {', '.join(strong) or 'none'}.",
        f"Traits you want that {subj} falls short on: {', '.join(gap) or 'none'}.",
    ]
    db = _dealbreakers(r)
    lines.append("Dealbreakers/heads-up: " + (" ".join(db) if db else "none"))
    return "\n".join(lines)

NOTE_SYSTEM = (
    "You're the user's blunt-but-warm friend giving a quick read on a potential match. "
    "Return ONLY a JSON object with these keys, each a SHORT casual blurb (1-2 sentences) spoken "
    "directly TO the user as 'you':\n"
    "- common_ground: what you'd click on together (the shared interests).\n"
    "- friction: what might annoy you - things YOU like that he doesn't, and things HE likes that "
    "you don't. Keep the direction exactly right; do not swap who likes what.\n"
    "- fit: how well he matches the traits you care about.\n"
    "- heads_up: real dealbreakers only (mismatched goals, big age gap, strong same-background "
    "need). If the brief says 'none', set this to an empty string \"\".\n"
    "- verdict: a quick gut call (worth a shot / maybe not / your call).\n"
    "Refer to the other person only by the given pronoun - never an id, number, or name. You are "
    "NOT on the date and not part of the couple: never say 'we', 'us', or 'our'. Use ONLY the facts "
    "provided - never invent interests, traits, brands, shows, or details. Casual texting-a-friend "
    "tone. No markdown, no emoji."
)

def _ban_hit(text):
    t = text.lower()
    hits = [n for n in BANNED_NAMES if n in t]
    if _BAD_PRON.search(text):
        hits.append("first-person-plural")
    return hits

def _note_call(brief, seed, timeout=180):
    payload = {"model": NOTE_MODEL, "stream": False, "format": "json",
               "options": {**NOTE_OPTS, "seed": int(seed)},
               "messages": [{"role": "system", "content": NOTE_SYSTEM},
                            {"role": "user", "content": brief}]}
    req = urllib.request.Request(OLLAMA_URL, data=json.dumps(payload).encode("utf-8"),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        content = json.loads(resp.read())["message"]["content"]
    try:
        obj = json.loads(content)
    except json.JSONDecodeError:
        return {"verdict": sanitize(content)}
    return {k: sanitize(str(obj.get(k, ""))) for k in SECTIONS}

def render_note(d):
    """Print each non-empty section under its little header."""
    for key in SECTIONS:
        text = d.get(key, "").strip()
        if text:
            print(SECTION_LABELS[key])
            print(f"  {text}\n")

# peek at a brief
print(match_brief(compatibility(*demo_pairs[0])))

The other person is male; refer to him as 'he'.
Interests you and he both enjoy: playing sports, dining out, reading, movies, concerts, music.
Things YOU like but he does NOT: exercising, watching TV.
Things HE likes but YOU do NOT: watching sports, museums & galleries.
Things you and he both can't stand: theater, yoga & meditation.
Traits you want that he is strong on: sincere, intelligent, attractive.
Traits you want that he falls short on: none.
Dealbreakers/heads-up: You're after 'to meet new people' while he wants 'a fun night out'.


In [7]:
# --- matchmaker_note: compatibility -> brief -> sectioned casual note (retry + heads_up fallback) ---
def matchmaker_note(a, b, max_tries=3):
    r = compatibility(a, b)
    brief = match_brief(r)
    seed0 = int(a) * 1000 + int(b)
    chosen = {}
    for t in range(max_tries):
        chosen = _note_call(brief, seed0 + 1000 * t)
        blob = " ".join(chosen.values())
        if blob.strip() and not _ban_hit(blob):
            break
    # guarantee real dealbreakers surface even if the model dropped heads_up
    db = _dealbreakers(r)
    if db and not chosen.get("heads_up", "").strip():
        chosen["heads_up"] = " ".join(db)
    return chosen

In [8]:
# --- Demo: sectioned casual matchmaker notes to A ---
for a, b in demo_pairs:
    print(f"===== to #{a} about #{b} =====")
    render_note(matchmaker_note(a, b))

===== to #1 about #11 =====


Common ground
  You two would click on sports days, trying new restaurants together and watching movies. He also digs concerts & music.

Friction
  He's not into exercising or TV shows like you are; you're probably over museums & galleries too - they clash there

Your type?
  The sincerity, intelligence & attractiveness scores seem spot-on for you!

Heads-up
  One goal mismatch: he wants a night out vs your focus on meeting new people

===== to #2 about #16 =====


Common ground
  You two have a lot of similar interests - exercising, dining out, museums & galleries, art, reading, theater, movies, and concerts. You'll likely spend your free time doing things you both enjoy.

Friction
  He likes playing sports but that's not really your thing; he also enjoys shopping which doesn't interest you much either.

Your type?
  You'd say it looks like a great match - he seems intelligent and fun, just the way you're looking for someone to be.

Verdict
  This one's worth taking a shot on

===== to #4 about #40 =====


Common ground
  You'd have a blast together exploring museums and galleries during the day, then dancing at clubs or concerts in the evenings!

Friction
  He doesn't share your passion for exercising, which could lead to some conflicting priorities. On top of that, he loves watching sports but you're not into it.

Your type?
  You'd give him high marks on being sincere and intelligent - those traits are super important to you! However, attractive isn't as crucial so this might be a slight miss for you

Heads-up
  Be aware: his goal is more low-key socializing which may not exactly match your preference for 'a fun night out' vibe.

Verdict
  Definitely worth giving it a shot

